In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py as h5

import test_rig_control_model.common as co

plt.style.use("../FST.mplstyle")
plt.rcParams["font.size"] = 10
plt.rcParams.update({"text.usetex": True, "font.family": "Helvetica"})

In [ ]:
k1 = -770690.9261910986
k2 = 7.624228003861329
k3 = 0.0020417184101237

s_1 = 35.36
s_2 = 41.6667
s_max = 74.16666667

q_list = np.linspace(0.00009, 0.00014, 1000)

pump_curve_1 = np.array(
    [k1 * q**2 + k2 * q * s_1 + 1.01930 * k3 * s_1**2 for q in q_list]
)
pump_curve_2 = np.array(
    [k1 * q**2 + k2 * q * s_2 + 0.827 * k3 * s_2**2 for q in q_list]
)

In [ ]:
def get_run_names_from_file(h5file_path):
    top_level_groups = []

    def visitor(name, obj):
        if isinstance(obj, h5.Group):
            if "/" not in name and name != "models":
                top_level_groups.append(name)

    with h5.File(h5file_path, "r") as f:
        f.visititems(visitor)

    return top_level_groups


def get_exp_results(file_name, run_index, location=co.DATA_DIR):
    run_names = get_run_names_from_file(location / file_name)
    time_series = pd.read_hdf(location / file_name, f"/{run_names[run_index]}/simulation_results/time_series")
    return time_series


exp_df_c_35_75 = get_exp_results("system_1_constant_operating_point_3536_75.hdf5", 1)
exp_df_c_35_50 = get_exp_results(
    "system_1_constant_operating_point_3536_50.hdf5", 2
)
exp_df_c_41_75 = get_exp_results("system_1_constant_operating_point_4167_75.hdf5", 0)
exp_df_p_35_41 = get_exp_results("system_1_pump_speed_up_3536_to_4167.hdf5", 1)
exp_df_v_25s = get_exp_results("system_1_valve_closure_75_to_50_25s.hdf5", 0)
exp_df_v_5s = get_exp_results("system_1_valve_closure_75_to_50_5s.hdf5", 0)

In [ ]:
n = 200
print("OP 35/75")
print("q: ", exp_df_c_35_75["test_rig.consumer_1_volume_flow"][n:].mean())
print("h: ", exp_df_c_35_75["test_rig.pump_1_pressure"][n:].mean() / (1000 * 9.81))

print("OP 35/50")
print("q: ", exp_df_c_35_50["test_rig.consumer_1_volume_flow"][n:].mean())
print("h: ", exp_df_c_35_50["test_rig.pump_1_pressure"][n:].mean() / (1000 * 9.81))

print("OP 41/75")
print("q: ", exp_df_c_41_75["test_rig.consumer_1_volume_flow"][n:].mean())
print("h: ", exp_df_c_41_75["test_rig.pump_1_pressure"][n:].mean() / (1000 * 9.81))


In [ ]:
x1 = 0.000110859
y1 = 2.62214

x2 = 0.00010191
y2 = 2.59466

x3 = 0.00012773
y3 = 3.1192138905187794

# a = k1
# b = ((y2 - k1 * x2**2) - (y1 - k1 * x1**2)) / (x2 - x1)
# c = y1 - k1 * x1**2 - b * x1

# nk1 = a
# nk2 = b / s_1
# nk3 = c / s_1**2


nk1, nk2, nk3 = 1.0, 57.45648878, 0.06786977

print(nk1)
print(nk2)
print(nk3)

In [ ]:
new_pump_curve_1 = np.array([nk1 * q**2 + nk2 * q * s_1 + nk3 * s_1 for q in q_list])
new_pump_curve_2 = np.array([nk1 * q**2 + nk2 * q * s_2 + nk3 * s_2 for q in q_list])

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

(l0,) = ax.plot(q_list, new_pump_curve_1, c="k", label="pump curves")
ax.plot(q_list, new_pump_curve_2, c="k", label="pump curve")


(l1,) = ax.plot(
    q_list,
    [
        (8 * 7.25 * 0.10367) / (np.pi**2 * 9.81 * 0.015**5) * q**2 + 1.61908
        # 1.619080192380957
        for q in q_list
    ],  # 1.1767
    c="dimgrey",
    linestyle="--",
    label="EWCM-DE plant curve, COP 35/75",
)

(l2,) = ax.plot(
    q_list,
    [
        (8 * 7.25 * 0.10367) / (np.pi**2 * 9.81 * 0.015**5) * q**2 + 1.61908 + 0.135
        # 1.619080192380957
        for q in q_list
    ],  # 1.1767
    c="darkgrey",
    linestyle="--",
    label="EWCM-DE plant curve, COP 35/50",
)


(l3,) = ax.plot(
    q_list,
    [
        (8 * 7.25 * 0.1035) / (np.pi**2 * 9.81 * 0.015**5) * q**2
        + 1.61908
        + 0.185
        + 0.05
        # 1.619080192380957
        for q in q_list
    ],  # 1.1767
    c="darkgrey",
    linestyle=":",
    label="EWCM-DE plant curve, COP 41/50",
)

(l4,) = ax.plot(
    q_list,
    [
        (8 * 7.25 * 0.1035) / (np.pi**2 * 9.81 * 0.015**5) * q**2 + 1.61908 + 0.185
        # 1.619080192380957
        for q in q_list
    ],  # 1.1767
    c="dimgrey",
    linestyle=":",
    label="EWCM-DE plant curve, COP 41/75",
)


ax.scatter(x1, y1, marker="+", c="k", linewidth=1.5, s=100, zorder=100)
ax.scatter(x2, y2, marker="+", c="k", linewidth=1.5, s=100, zorder=100)
ax.scatter(x3, y3, marker="+", c="k", linewidth=1.5, s=100, zorder=100)
ax.scatter(
    x3 - 0.000003, y3 + 0.01, marker="+", c="k", linewidth=1.5, s=100, zorder=100
)

ax.annotate(
    "COP 35/75, exp.",
    xy=(x1, y1),
    xycoords="data",
    xytext=(25, -15),
    textcoords="offset points",
    # arrowprops=dict(facecolor="black", width=0.1, headlength=0.001),
    horizontalalignment="center",
    verticalalignment="bottom",
)

ax.annotate(
    "COP 35/50, exp.",
    xy=(x2, y2),
    xycoords="data",
    xytext=(5, -15),
    textcoords="offset points",
    # arrowprops=dict(facecolor="black", width=0.1, headlength=0.001),
    horizontalalignment="center",
    verticalalignment="bottom",
)

ax.annotate(
    "COP 41/75, exp.",
    xy=(x3, y3),
    xycoords="data",
    xytext=(25, -15),
    textcoords="offset points",
    # arrowprops=dict(facecolor="black", width=0.1, headlength=0.001),
    horizontalalignment="center",
    verticalalignment="bottom",
    fontsize=8,
)

ax.annotate(
    "COP 41/50, est.",
    xy=(x3 - 0.000001, y3),
    xycoords="data",
    xytext=(-30, 6),
    textcoords="offset points",
    # arrowprops=dict(facecolor="black", width=0.1, headlength=0.001),
    horizontalalignment="center",
    verticalalignment="bottom",
)

ax.annotate(
    "$\omega=35.36\mathrm{Hz}$",
    xy=(0.00013, 2.6),
    xycoords="data",
    # xytext=(-25, 5),
    textcoords="offset points",
    # arrowprops=dict(facecolor="black", width=0.1, headlength=0.001),
    horizontalalignment="center",
    verticalalignment="bottom",
    fontsize=8,
    # backgroundcolor="1.0",
)

ax.annotate(
    "$\omega=41.67\mathrm{Hz}$",
    xy=(0.0001, 3),
    xycoords="data",
    # xytext=(-25, 5),
    textcoords="offset points",
    # arrowprops=dict(facecolor="black", width=0.1, headlength=0.001),
    horizontalalignment="center",
    verticalalignment="bottom",
    fontsize=8,
    # backgroundcolor="1.0",
)

ax.set_xlim((0.000095, 0.000135))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

ax.set_ylim()
ax.legend(handles=[l0, l2, l1, l3, l4], fontsize=7)

ax.set_xlabel("VOLUME FLOW in $\mathrm{m}^3\mathrm{s}^{-1}$")
ax.set_ylabel("PRESSURE in $\mathrm{m}$")

# fig.savefig("fig_pump_curves.pdf", bbox_inches="tight")
fig.savefig("fig_pump_curves_COP.svg", bbox_inches="tight")
fig.savefig("fig_pump_curves_COP.png", bbox_inches="tight")